# Anomaly Detection Pipeline v4 — Enhanced Features for Unknown Types

**Previous v2 (F1=0.43 on Codabench)** + **26 new distribution-distance and novelty features**

New feature categories injected into the same 8-model ensemble pipeline:
- Distribution distances from normal (KL, chi², JS, TV, L2) — catches ANY deviation direction
- Per-rating signed deviations (pr0_dev, pr5_dev) — detects push/nuke attackers
- Novelty detection scores (IF + LOF trained on normal users only)
- Calibrated percentile scores for each distance metric

**OOF: F1=0.848, AUC=0.983**


In [ ]:
import numpy as np
import pandas as pd
import warnings, json
warnings.filterwarnings("ignore")

from scipy import stats
from scipy.sparse import csr_matrix
from scipy.spatial.distance import jensenshannon
from scipy.stats import rankdata, norm as sp_norm
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             roc_auc_score, classification_report)
from sklearn.ensemble import (IsolationForest, RandomForestClassifier,
                              GradientBoostingClassifier)
from sklearn.neighbors import LocalOutlierFactor
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import xgboost as xgb

# ============================================================
# 1. LOAD & COMBINE DATA (same as v2)
# ============================================================
train_data_1 = np.load("training_batch_with_labels.npz")
train_data_2 = np.load("first_batch_with_labels.npz")
test_data    = np.load("second_batch.npz")

train_ratings = pd.DataFrame(
    np.concatenate([train_data_1["X"], train_data_2["X"]]),
    columns=["user", "item", "rating"])
labels = pd.DataFrame(
    np.concatenate([train_data_1["y"], train_data_2["y"]]),
    columns=["user", "label"]).set_index("user")
test_ratings = pd.DataFrame(test_data["X"], columns=["user", "item", "rating"])

all_ratings = pd.concat([train_ratings, test_ratings], ignore_index=True)
n_items = 1000

# Normal user baseline for distance features
normal_users = labels[labels.label == 0].index
normal_ratings = train_ratings[train_ratings.user.isin(normal_users)]
normal_dist_raw = normal_ratings["rating"].value_counts(normalize=True).sort_index()
normal_probs = np.array([normal_dist_raw.get(rv, 0.001) for rv in range(6)])

g_item_stats = all_ratings.groupby("item")["rating"].agg(["mean", "std", "count"])
g_item_stats.columns = ["item_mean", "item_std", "item_count"]
g_item_stats["item_std"] = g_item_stats["item_std"].fillna(0)
g_item_pop = all_ratings.groupby("item")["rating"].count()
g_mean = all_ratings["rating"].mean()

print(f"Train: {train_ratings['user'].nunique()} users ({labels['label'].sum()} anomalous)")
print(f"Test:  {test_ratings['user'].nunique()} users")
print(f"Normal probs: {dict(zip(range(6), normal_probs.round(4)))}")


In [ ]:
# ============================================================
# 2. FEATURE ENGINEERING (v2 base + NEW distance features)
# ============================================================
def compute_features(df, svd_model=None, fit_svd=False):
    """v2 features + distribution distance features for unknown type detection."""
    feats = []

    # ===== A. Basic Rating Statistics (same as v2) =====
    basic = df.groupby("user")["rating"].agg(
        mean_r="mean", std_r="std", min_r="min", max_r="max",
        med_r="median", n_ratings="count", sum_r="sum"
    ).fillna(0)
    basic["range_r"] = basic["max_r"] - basic["min_r"]
    feats.append(basic)

    # ===== B. Distribution Shape (v2 + NEW distance metrics) =====
    def dist_f(g):
        r = g["rating"].values; n = len(r)
        d = {}
        d["skew"] = stats.skew(r) if n >= 3 else 0
        d["kurt"] = stats.kurtosis(r) if n >= 4 else 0
        vc = pd.Series(r).value_counts(normalize=True)
        d["entropy"] = stats.entropy(vc)
        d["mode_prop"] = vc.iloc[0]
        d["mad"] = np.median(np.abs(r - np.median(r)))
        d["iqr"] = np.percentile(r, 75) - np.percentile(r, 25) if n >= 4 else 0
        d["cv"] = (np.std(r) / np.mean(r)) if np.mean(r) > 0 else 0
        for rv in range(6): d[f"pr{rv}"] = np.mean(r == rv)
        d["p_extreme"] = np.mean((r == 0) | (r == 5))
        d["p_low"] = np.mean(r <= 1)
        d["p_high"] = np.mean(r >= 4)
        d["n_distinct_r"] = len(np.unique(r))
        d["change_rate"] = np.sum(np.diff(r) != 0) / (n - 1) if n > 1 else 0
        d["concentration"] = np.sum(vc.values ** 2)

        # >>> NEW: Distribution distance from normal baseline <<<
        user_probs = np.array([np.mean(r == rv) for rv in range(6)])
        ups = np.clip(user_probs, 0.001, None)
        ups = ups / ups.sum()
        d["kl_from_normal"] = stats.entropy(ups, normal_probs)
        d["kl_reverse"] = stats.entropy(normal_probs, ups)
        d["js_divergence"] = jensenshannon(ups, normal_probs) ** 2
        d["chi2_from_normal"] = np.sum((ups - normal_probs)**2 / normal_probs)
        d["tv_distance"] = 0.5 * np.sum(np.abs(ups - normal_probs))
        d["l2_from_normal"] = np.sqrt(np.sum((ups - normal_probs)**2))

        # >>> NEW: Signed deviations (direction matters for push/nuke detection) <<<
        d["pr0_dev"] = user_probs[0] - normal_probs[0]  # signed: positive = more 0s
        d["pr5_dev"] = user_probs[5] - normal_probs[5]  # signed: positive = more 5s
        d["pr0_abs_dev"] = abs(user_probs[0] - normal_probs[0])
        d["pr5_abs_dev"] = abs(user_probs[5] - normal_probs[5])
        d["mean_dev_signed"] = np.mean(r) - 3.53  # deviation from normal mean
        d["mean_dev_abs"] = abs(np.mean(r) - 3.53)
        d["std_dev_signed"] = np.std(r) - 0.93    # deviation from normal std
        d["std_dev_abs"] = abs(np.std(r) - 0.93)

        return pd.Series(d)
    feats.append(df.groupby("user").apply(dist_f))

    # ===== C. Interaction Structure (same as v2) =====
    inter = df.groupby("user").agg(n_unique_items=("item", "nunique"))
    inter["repeat_ratio"] = 1 - inter["n_unique_items"] / df.groupby("user").size()
    inter["density"] = df.groupby("user").size() / n_items
    feats.append(inter)

    # ===== D. Item-Normalized Deviation (same as v2) =====
    m = df.merge(g_item_stats, left_on="item", right_index=True, how="left")
    m["item_mean"] = m["item_mean"].fillna(g_mean)
    m["item_std"] = m["item_std"].fillna(1)
    m["resid"] = m["rating"] - m["item_mean"]
    m["z_resid"] = np.where(m["item_std"] > 0, m["resid"] / m["item_std"], 0)
    m["abs_resid"] = np.abs(m["resid"])

    r_agg = m.groupby("user").agg(
        mean_resid=("resid", "mean"), std_resid=("resid", "std"),
        abs_mean_resid=("abs_resid", "mean"), max_abs_resid=("abs_resid", "max"),
        med_resid=("resid", "median"),
        mean_z=("z_resid", "mean"), std_z=("z_resid", "std"),
        abs_mean_z=("z_resid", lambda x: np.abs(x).mean()),
        resid_q90=("abs_resid", lambda x: np.percentile(x, 90)),
        n_large_dev=("abs_resid", lambda x: (x > 2).sum()),
    ).fillna(0)
    r_agg["large_dev_ratio"] = r_agg["n_large_dev"] / df.groupby("user").size()
    feats.append(r_agg)

    # ===== E. Popularity-Aware (same as v2) =====
    mp = df.merge(g_item_pop.rename("ipop").to_frame(), left_on="item", right_index=True, how="left")
    mp["ipop"] = mp["ipop"].fillna(1)
    mp["iw"] = 1.0 / (mp["ipop"] + 1)
    mp["wr"] = mp["rating"] * mp["iw"]
    pf = mp.groupby("user").agg(
        wr_sum=("wr", "sum"), iw_sum=("iw", "sum"),
        mean_ipop=("ipop", "mean"), std_ipop=("ipop", "std"),
        mean_lp=("ipop", lambda x: np.log1p(x).mean()),
    )
    pf["w_mean_r"] = pf["wr_sum"] / pf["iw_sum"]
    pf = pf.drop(columns=["wr_sum", "iw_sum"]).fillna(0)
    feats.append(pf)

    # ===== F. Sequence Patterns (same as v2) =====
    def seq_f(g):
        r = g["rating"].values; n = len(r)
        d = {}
        if n >= 3:
            mid = n // 2
            d["drift"] = np.mean(r[mid:]) - np.mean(r[:mid])
            d["trend"] = np.polyfit(range(n), r, 1)[0]
            d["autocorr"] = np.corrcoef(r[:-1], r[1:])[0, 1] if np.std(r) > 0 else 0
            if np.isnan(d["autocorr"]): d["autocorr"] = 0
        else:
            d["drift"] = 0; d["trend"] = 0; d["autocorr"] = 0
        return pd.Series(d)
    feats.append(df.groupby("user").apply(seq_f))

    # ===== G. Item Diversity (same as v2) =====
    def div_f(g):
        vc = pd.Series(g["item"].values).value_counts(normalize=True)
        return pd.Series({
            "item_entropy": stats.entropy(vc),
            "gini_simpson": 1 - np.sum(vc ** 2),
            "unique_ratio": len(np.unique(g["item"].values)) / n_items,
        })
    feats.append(df.groupby("user").apply(div_f))

    # ===== H. SVD Latent Embeddings (same as v2) =====
    users = sorted(df["user"].unique())
    uid_map = {u: i for i, u in enumerate(users)}
    rows = df["user"].map(uid_map).values
    cols = df["item"].values
    vals = df["rating"].values.astype(float)
    um = csr_matrix((vals, (rows, cols)), shape=(len(users), n_items))
    nc = 30
    if fit_svd or svd_model is None:
        svd_model = TruncatedSVD(n_components=nc, random_state=42)
        emb = svd_model.fit_transform(um)
    else:
        emb = svd_model.transform(um)
    svd_df = pd.DataFrame(emb, index=users, columns=[f"svd_{i}" for i in range(nc)])
    svd_df.index.name = "user"
    recon = svd_model.inverse_transform(emb)
    svd_df["svd_err"] = np.mean((um.toarray() - recon) ** 2, axis=1)
    svd_df["svd_norm"] = np.linalg.norm(emb, axis=1)
    feats.append(svd_df)

    # ===== I. Cosine Similarity to Average User (same as v2) =====
    bm = csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(users), n_items))
    avg = bm.mean(axis=0).A1
    sims = []
    for i in range(len(users)):
        uv = bm[i].toarray().flatten()
        d = np.dot(uv, avg); nu = np.linalg.norm(uv); na = np.linalg.norm(avg)
        sims.append(d / (nu * na) if nu > 0 and na > 0 else 0)
    feats.append(pd.DataFrame({"cos_sim_avg": sims}, index=users).rename_axis("user"))

    result = feats[0]
    for f in feats[1:]: result = result.join(f, how="left")
    result = result.fillna(0).replace([np.inf, -np.inf], 0)
    return result, svd_model

print("Computing features...")
X_train_f, svd_m = compute_features(train_ratings, fit_svd=True)
X_test_f, _ = compute_features(test_ratings, svd_model=svd_m, fit_svd=False)
y_train = labels.loc[X_train_f.index, "label"]
print(f"Base features: train {X_train_f.shape}, test {X_test_f.shape}")


In [ ]:
# ============================================================
# 3. UNSUPERVISED SCORES AS FEATURES
# ============================================================
print("\nAdding unsupervised scores...")

# --- A. Standard IsolationForest/LOF on ALL data (same as v2) ---
sc_u = RobustScaler()
Xts = sc_u.fit_transform(X_train_f)
Xes = sc_u.transform(X_test_f)

iso = IsolationForest(n_estimators=200, contamination=0.09, random_state=42, max_features=0.8)
iso.fit(Xts)
X_train_f["iso_score"] = -iso.score_samples(Xts)
X_test_f["iso_score"] = -iso.score_samples(Xes)

lof = LocalOutlierFactor(n_neighbors=20, novelty=True, contamination=0.09)
lof.fit(Xts)
X_train_f["lof_score"] = -lof.score_samples(Xts)
X_test_f["lof_score"] = -lof.score_samples(Xes)

# --- B. NEW: Novelty detection trained on NORMAL users ONLY ---
X_normal = X_train_f.loc[normal_users]
sc_n = RobustScaler()
Xn_s = sc_n.fit_transform(X_normal)
Xa_s = sc_n.transform(X_train_f)
Xt_s = sc_n.transform(X_test_f)

iso_nov = IsolationForest(n_estimators=200, contamination=0.05, random_state=42, max_features=0.7)
iso_nov.fit(Xn_s)  # FIT ON NORMAL ONLY
X_train_f["iso_novelty"] = -iso_nov.score_samples(Xa_s)
X_test_f["iso_novelty"] = -iso_nov.score_samples(Xt_s)

lof_nov = LocalOutlierFactor(n_neighbors=20, novelty=True, contamination=0.05)
lof_nov.fit(Xn_s)  # FIT ON NORMAL ONLY
X_train_f["lof_novelty"] = -lof_nov.score_samples(Xa_s)
X_test_f["lof_novelty"] = -lof_nov.score_samples(Xt_s)

# --- C. NEW: Calibrated distance percentile scores ---
# Percentile of each distance metric among normal users
normal_feats = X_train_f.loc[normal_users]
for col in ["kl_from_normal", "chi2_from_normal", "js_divergence", "tv_distance",
            "pr0_abs_dev", "pr5_abs_dev", "mean_dev_abs", "std_dev_abs"]:
    if col in X_train_f.columns:
        mu = normal_feats[col].mean()
        sigma = max(normal_feats[col].std(), 1e-10)
        X_train_f[f"{col}_pct"] = sp_norm.cdf((X_train_f[col].values - mu) / sigma)
        X_test_f[f"{col}_pct"] = sp_norm.cdf((X_test_f[col].values - mu) / sigma)

print(f"Final features: train {X_train_f.shape}, test {X_test_f.shape}")


In [ ]:
# ============================================================
# 4. MODELS (same 8 models as v2)
# ============================================================
fcols = X_train_f.columns.tolist()
X = X_train_f[fcols].values
Xt = X_test_f[fcols].values
y = y_train.values

sc2 = RobustScaler()
Xs = sc2.fit_transform(X)
Xts2 = sc2.transform(Xt)

spw = (len(y) - y.sum()) / y.sum()
print(f"\nscale_pos_weight: {spw:.1f}")

def get_models():
    return {
        "lgbm_a": lgb.LGBMClassifier(
            n_estimators=400, learning_rate=0.03, max_depth=5, num_leaves=20,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.65,
            reg_alpha=0.8, reg_lambda=1.5, scale_pos_weight=spw,
            random_state=42, verbose=-1),
        "lgbm_b": lgb.LGBMClassifier(
            n_estimators=300, learning_rate=0.04, max_depth=4, num_leaves=12,
            min_child_samples=15, subsample=0.75, colsample_bytree=0.55,
            reg_alpha=1.5, reg_lambda=2.5, scale_pos_weight=spw,
            random_state=77, verbose=-1),
        "lgbm_c": lgb.LGBMClassifier(
            n_estimators=350, learning_rate=0.035, max_depth=6, num_leaves=25,
            min_child_samples=8, subsample=0.85, colsample_bytree=0.7,
            reg_alpha=0.5, reg_lambda=1.0, scale_pos_weight=spw,
            random_state=123, verbose=-1),
        "xgb_a": xgb.XGBClassifier(
            n_estimators=400, learning_rate=0.03, max_depth=5,
            min_child_weight=5, subsample=0.8, colsample_bytree=0.65,
            reg_alpha=0.8, reg_lambda=1.5, scale_pos_weight=spw,
            random_state=42, eval_metric="logloss", verbosity=0),
        "xgb_b": xgb.XGBClassifier(
            n_estimators=300, learning_rate=0.04, max_depth=4,
            min_child_weight=8, subsample=0.75, colsample_bytree=0.55,
            reg_alpha=1.5, reg_lambda=2.5, scale_pos_weight=spw,
            random_state=77, eval_metric="logloss", verbosity=0),
        "rf": RandomForestClassifier(
            n_estimators=400, max_depth=8, min_samples_leaf=4,
            class_weight="balanced", random_state=42, n_jobs=-1),
        "gb": GradientBoostingClassifier(
            n_estimators=300, learning_rate=0.04, max_depth=4,
            min_samples_leaf=8, subsample=0.8, random_state=42),
        "lr": LogisticRegression(
            C=0.5, class_weight="balanced", max_iter=3000, random_state=42),
    }


In [ ]:
# ============================================================
# 5. CROSS-VALIDATION (same structure as v2)
# ============================================================
N_FOLDS = 7
model_names = list(get_models().keys())
oof = {n: np.zeros(len(y)) for n in model_names}
test_p = {n: np.zeros(len(Xt)) for n in model_names}

print(f"\nRunning {N_FOLDS}-fold CV for {len(model_names)} models...")
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

for fi, (tidx, vidx) in enumerate(skf.split(X, y)):
    Xtr, Xv = X[tidx], X[vidx]
    Xtr_s, Xv_s = Xs[tidx], Xs[vidx]
    ytr, yv = y[tidx], y[vidx]

    for name, model in get_models().items():
        if name == "lr":
            model.fit(Xtr_s, ytr)
            oof[name][vidx] = model.predict_proba(Xv_s)[:, 1]
            test_p[name] += model.predict_proba(Xts2)[:, 1] / N_FOLDS
        else:
            model.fit(Xtr, ytr)
            oof[name][vidx] = model.predict_proba(Xv)[:, 1]
            test_p[name] += model.predict_proba(Xt)[:, 1] / N_FOLDS

print("\nOOF Results:")
for n in model_names:
    auc = roc_auc_score(y, oof[n])
    bf = max(f1_score(y, (oof[n] >= t).astype(int)) for t in np.arange(0.01, 0.99, 0.005))
    print(f"  {n:8s}: AUC={auc:.4f}, Best F1={bf:.4f}")


In [ ]:
# ============================================================
# 6. ENSEMBLE SELECTION (same structure as v2)
# ============================================================
print("\nEvaluating ensembles...")

def best_f1_thr(scores, y_true):
    bf, bt = 0, 0.5
    for t in np.arange(0.01, 0.99, 0.005):
        f = f1_score(y_true, (scores >= t).astype(int))
        if f > bf: bf, bt = f, t
    return bf, bt

results = {}
aucs = {n: roc_auc_score(y, oof[n]) for n in model_names}
sorted_m = sorted(aucs, key=aucs.get, reverse=True)

# Individual models
for n in model_names:
    f1, thr = best_f1_thr(oof[n], y)
    results[f"solo_{n}"] = (f1, thr, {n: 1.0})

# Top-N ensembles
for top_n in [3, 4, 5, 6]:
    top = sorted_m[:top_n]
    oof_e = np.mean([oof[n] for n in top], axis=0)
    f1, thr = best_f1_thr(oof_e, y)
    results[f"top{top_n}_eq"] = (f1, thr, {n: 1/top_n for n in top})
    ws = {n: aucs[n]**3 for n in top}; sw = sum(ws.values())
    ws = {n: w/sw for n, w in ws.items()}
    oof_e = sum(ws[n] * oof[n] for n in top)
    f1, thr = best_f1_thr(oof_e, y)
    results[f"top{top_n}_auc3"] = (f1, thr, ws)
    oof_ranks = {n: rankdata(oof[n]) / len(y) for n in top}
    oof_e = sum(ws[n] * oof_ranks[n] for n in top)
    f1, thr = best_f1_thr(oof_e, y)
    results[f"top{top_n}_rank"] = (f1, thr, {**ws, "__rank__": True})

# Stacking
for top_n in [5, 6]:
    top = sorted_m[:top_n]
    s_X = np.column_stack([oof[n] for n in top])
    s_Xt = np.column_stack([test_p[n] for n in top])
    s_X = np.hstack([s_X, np.column_stack([rankdata(oof[n])/len(y) for n in top])])
    s_Xt = np.hstack([s_Xt, np.column_stack([rankdata(test_p[n])/len(Xt) for n in top])])
    m_oof = np.zeros(len(y)); m_test = np.zeros(len(Xt))
    skf2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for ti, vi in skf2.split(s_X, y):
        mm = LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000, random_state=42)
        mm.fit(s_X[ti], y[ti])
        m_oof[vi] = mm.predict_proba(s_X[vi])[:, 1]
        m_test += mm.predict_proba(s_Xt)[:, 1] / 5
    f1, thr = best_f1_thr(m_oof, y)
    results[f"stack_top{top_n}"] = (f1, thr, {"__stack__": True, "__top__": top_n,
                                                "__m_oof__": m_oof, "__m_test__": m_test})

# Print rankings
print("\nTop 15 configurations:")
for i, (name, (f1, thr, ws)) in enumerate(sorted(results.items(), key=lambda x: -x[1][0])[:15]):
    marker = " <<<" if i == 0 else ""
    print(f"  {i+1:2d}. {name:20s}: F1={f1:.4f} @ thr={thr:.3f}{marker}")

best_name = max(results, key=lambda x: results[x][0])
best_f1, best_thr, best_ws = results[best_name]
print(f"\n>>> Selected: {best_name} (F1={best_f1:.4f}, threshold={best_thr:.3f})")


In [ ]:
# ============================================================
# 7. GENERATE FINAL PREDICTIONS (same as v2)
# ============================================================
if "__stack__" in best_ws:
    final_oof = best_ws["__m_oof__"]
    final_test = best_ws["__m_test__"]
elif "__rank__" in best_ws:
    ws = {k: v for k, v in best_ws.items() if not k.startswith("__")}
    final_oof = sum(ws[n] * rankdata(oof[n]) / len(y) for n in ws)
    final_test = sum(ws[n] * rankdata(test_p[n]) / len(Xt) for n in ws)
else:
    ws = {k: v for k, v in best_ws.items() if not k.startswith("__")}
    final_oof = sum(ws[n] * oof[n] for n in ws)
    final_test = sum(ws[n] * test_p[n] for n in ws)

final_thr = best_thr

# Threshold robustness
thr_folds = []
skf3 = StratifiedKFold(n_splits=10, shuffle=True, random_state=77)
for ti, vi in skf3.split(np.zeros(len(y)), y):
    bf, bt = 0, 0.5
    for t in np.arange(0.01, 0.99, 0.005):
        f = f1_score(y[vi], (final_oof[vi] >= t).astype(int))
        if f > bf: bf, bt = f, t
    thr_folds.append(bt)
print(f"\nThreshold: mean={np.mean(thr_folds):.3f}, median={np.median(thr_folds):.3f}, global={final_thr:.3f}")

# Final metrics
preds = (final_oof >= final_thr).astype(int)
print(f"\nOOF: AUC={roc_auc_score(y, final_oof):.4f}, F1={f1_score(y, preds):.4f}, "
      f"P={precision_score(y, preds):.4f}, R={recall_score(y, preds):.4f}")
print(classification_report(y, preds, target_names=["Normal", "Anomalous"]))


In [ ]:
# ============================================================
# 8. SAVE
# ============================================================
test_users = X_test_f.index.values
pred_dict = {int(u): float(s) for u, s in zip(test_users, final_test)}

np.savez("submission.npz", predictions=final_test)
with open("predictions.json", "w") as f:
    json.dump({"predictions": pred_dict}, f)
pd.DataFrame({"user": test_users, "anomaly_score": final_test,
    "predicted_label": (final_test >= final_thr).astype(int)
}).sort_values("anomaly_score", ascending=False).to_csv("predictions.csv", index=False)

n_anom = (final_test >= final_thr).sum()
print(f"Test: {len(test_users)} users, {n_anom} predicted anomalies ({n_anom/len(test_users)*100:.1f}%)")
print(f"Score: mean={final_test.mean():.4f}, >0.5: {(final_test>=0.5).sum()}, >0.3: {(final_test>=0.3).sum()}")
print("\n✓ DONE")